<a href="https://colab.research.google.com/github/SimplyBecca5220/NAFDAC-Drug-Verification-Risk-Classifier/blob/main/3mtt_nafdac.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from google.colab import drive

# Step 1: Mount Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
file_path = '/content/drive/MyDrive/Datasets /merged_table.csv'

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Datasets /merged_table.csv', sep=';', on_bad_lines='skip')
df.head()

,S/N,"""SAMPLE CODE""","""LABORATORY NUMBER""","""STATE""","""BRAND NAME""","""GENERIC NAME""","""PRODUCT CLASS""","""NAFDAC REG. NO""","""DOSAGE FORM""","""STRENGTH""",...,"""MARKET AUTHORISATION HOLDER""","""MARKET AUTHORISATION HOLDER ADDRESS""","""MANUFACTURER""","""MANUFACTURER ADDRESS""","""COUNTRY OF ORIGIN""","""PHYSICAL DESCRIPTION""","""SPECIFICATION""","""COMMENT""","""REMARKS""","""YEAR CAPTURED"""
0,1,"""ABI/ADG/001/211123""","""KLS/AB/PMS/DCN/0025/24""","""ABIA""","""MEPIRYL""","""GLIMEPIRIDE"" ...","""ANTIDIABETIC""","""04-9439""","""TABLET""","""4MG""",...,"""MAY & BAKER NIGERIA PLC"" ...","""1, MAY & BAKER AVENUE, OFF IDIROKO ROAD, (OP...","""MAY & BAKER NIGERIA PLC"" ...","""1, MAY & BAKER AVENUE, OFF IDIROKO ROAD, (OP...","""NIGERIA""","""A RECTANGULAR BOX WITH WHITE AND BLUE BACKG...","""NIL""","""NIL"" ...","""PASS"" ...","""2023"""
1,2,"""ABI/ADG/002/211123""","""KLS/AB/PMS/DCN/0026/24""","""ABIA""","""AMARYL""","""GLIMEPIRIDE"" ...","""ANTIDIABETIC""","""04-2852""","""TABLET""","""4MG""",...,"""SANOFI - AVENTIS DEUTSCHLAND GMBH"" ...","""D-65926 FRANKFURT AM MAIN, GERMANY"" ...","""SANOFI S.P.A"" ...","""STRADA STATALE 17, KM 22 67019 SCOPPITO (QA)...","""ITALY""","""A RECTANGULAR BOX WITH WHITE AND BLUE BACKG...","""NIL""","""NIL"" ...","""PASS"" ...","""2023"""
2,3,"""ABI/ADG/003/221123""","""KLS/AB/PMS/DCN/0027/24""","""ABIA""","""MEPIRYL""","""GLIMEPIRIDE"" ...","""ANTIDIABETIC""","""04-9605""","""TABLET""","""4MG""",...,"""MAY & BAKER NIGERIA PLC"" ...","""1, MAY & BAKER AVENUE, OFF IDIROKO ROAD, (OP...","""MAY & BAKER NIGERIA PLC"" ...","""1, MAY & BAKER AVENUE, OFF IDIROKO ROAD, (OP...","""NIGERIA""","""A RECTANGULAR BOX WITH WHITE AND BLUE BACK...","""NIL""","""NIL"" ...","""PASS"" ...","""2023"""
3,4,"""ABI/ADG/004/221123""","""KLS/AB/PMS/DCN/0028/24""","""ABIA""","""MEPIRYL""","""GLIMEPIRIDE"" ...","""ANTIDIABETIC""","""04-9605""","""TABLET""","""4MG""",...,"""MAY & BAKER NIGERIA PLC"" ...","""1, MAY & BAKER AVENUE, OFF IDIROKO ROAD, (OP...","""MAY & BAKER NIGERIA PLC"" ...","""1, MAY & BAKER AVENUE, OFF IDIROKO ROAD, (OP...","""NIGERIA""","""A RECTANGULAR BOX WITH WHITE AND BLUE BACK...","""NIL""","""NIL"" ...","""PASS"" ...","""2023"""
4,5,"""ABI/ADG/005/231123""","""KLS/AB/PMS/DCN/0029/24""","""ABIA""","""GETRYL""","""GLIMEPIRIDE"" ...","""ANTIDIABETIC""","""A4-8747""","""TABLET""","""4MG""",...,"""GETZ PHARMA (PVT) LIMITED"" ...","""29-30/27, KORANGI INDUSTRIAL AREA, KARACHI -...","""GETZ PHARMA (PVT) LIMITED"" ...","""29-30/27, KORANGI INDUSTRIAL AREA, KARACHI -...","""PAKISTAN""","""A RECTANGULAR BOX WITH WHITE PINK, AND YELLO...","""NIL""","""NIL"" ...","""PASS"" ...","""2023"""


In [ ]:
# ==========================================
# 1. SETUP & DATA CLEANING
# ==========================================
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Load CSV handling custom delimitation and double quotes
df = pd.read_csv('/content/drive/MyDrive/Datasets /merged_table.csv', sep=r'\s*;\s*', engine='python', quotechar='"')

# Clean headers and remove excess quote characters
df.columns = [col.strip(' "') for col in df.columns]

text_cols = ['BRAND NAME', 'GENERIC NAME', 'NAFDAC REG. NO', 'MANUFACTURER', 'COUNTRY OF ORIGIN', 'REMARKS', 'COMMENT']
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip(' "')

# Encode target label: 1 = Suspicious/Unsatisfactory, 0 = Genuine/Pass
def encode_target(remark):
    r = str(remark).upper()
    if 'UNSATISFACTORY' in r or 'FAIL' in r:
        return 1
    elif 'PASS' in r or 'SATISFACTORY' in r:
        return 0
    return np.nan

df['is_suspicious'] = df['REMARKS'].apply(encode_target)

# Filter out unlabelled rows
dataset = df.dropna(subset=['is_suspicious']).copy()
dataset['is_suspicious'] = dataset['is_suspicious'].astype(int)

print(f"Dataset Loaded Successfully! Total Samples: {len(dataset)}")
print("Class Breakdown:")
print(f" - Genuine (0): {sum(dataset['is_suspicious'] == 0)}")
print(f" - Suspicious/Unsatisfactory (1): {sum(dataset['is_suspicious'] == 1)}")

# ==========================================
# 2. HYBRID MODEL: LOOKUP + ML CLASSIFIER
# ==========================================

# Feature text creation for ML model
dataset['combined_features'] = (
    dataset['BRAND NAME'].fillna('') + ' ' +
    dataset['GENERIC NAME'].fillna('') + ' ' +
    dataset['NAFDAC REG. NO'].fillna('') + ' ' +
    dataset['MANUFACTURER'].fillna('') + ' ' +
    dataset['COUNTRY OF ORIGIN'].fillna('')
)

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    dataset['combined_features'],
    dataset['is_suspicious'],
    test_size=0.20,
    random_state=42,
    stratify=dataset['is_suspicious']
)

# Text Vectorization
vectorizer = TfidfVectorizer(max_features=500, ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Classifier training with class weighting for imbalanced data
model = RandomForestClassifier(class_weight='balanced', random_state=42, n_estimators=100)
model.fit(X_train_vec, y_train)

# Model Performance Metrics
y_pred = model.predict(X_test_vec)
print("\n" + "="*40)
print("  CLASSIFICATION REPORT (ML BASELINE)")
print("="*40)
print(classification_report(y_test, y_pred, target_names=['Genuine', 'Suspicious']))


# ==========================================
# 3. VERIFICATION ENGINE FUNCTION
# ==========================================

def verify_drug_product(nafdac_no, brand_name, manufacturer=""):
    """
    Checks incoming drug details against known NAFDAC registry entries
    and runs a secondary ML risk scoring check.
    """
    nafdac_no = nafdac_no.strip()
    brand_name = brand_name.strip()

    # 1. Exact Registry Match Check
    registry_matches = dataset[dataset['NAFDAC REG. NO'].str.upper() == nafdac_no.upper()]

    if not registry_matches.empty:
        # Check if brand matches registered brand
        brand_matches = registry_matches[registry_matches['BRAND NAME'].str.upper() == brand_name.upper()]

        if not brand_matches.empty:
            sample_status = brand_matches['REMARKS'].values[0]
            is_fail = brand_matches['is_suspicious'].values[0]

            if is_fail == 1:
                return {
                    'status': 'SUSPICIOUS',
                    'confidence': 1.0,
                    'reason': f"NAFDAC Reg No found in surveillance records with flagged status: '{sample_status}'."
                }
            return {
                'status': 'GENUINE',
                'confidence': 0.98,
                'reason': "Matches valid NAFDAC registered product in surveillance database."
            }
        else:
            registered_brands = list(registry_matches['BRAND NAME'].unique())
            return {
                'status': 'SUSPICIOUS',
                'confidence': 0.90,
                'reason': f"NAFDAC Reg No exists, but is registered to {registered_brands}, not '{brand_name}'."
            }

    # 2. Machine Learning Fallback for Unregistered / New Entries
    input_text = f"{brand_name} {nafdac_no} {manufacturer}"
    input_vec = vectorizer.transform([input_text])
    risk_prob = model.predict_proba(input_vec)[0][1]

    if risk_prob > 0.40:
        return {
            'status': 'SUSPICIOUS',
            'confidence': round(float(risk_prob), 2),
            'reason': "Unregistered NAFDAC ID combined with high-risk manufacturer/text pattern score."
        }

    return {
        'status': 'UNVERIFIED / REQUIRES REVIEW',
        'confidence': round(1 - float(risk_prob), 2),
        'reason': "NAFDAC number not found in current surveillance records. Proceed with caution."
    }

# ==========================================
# 4. DEMO RUN
# ==========================================
print("\n--- SAMPLE TEST CASES ---")

# Case 1: Known Pass
test1 = verify_drug_product("04-9439", "MEPIRYL")
print("\nTest 1 (Valid Record):", test1)

# Case 2: Mismatched Brand & NAFDAC
test2 = verify_drug_product("04-9439", "FAKE AMARYL")
print("\nTest 2 (Mismatched Brand):", test2)

# Case 3: Flagged Unsatisfactory Drug
test3 = verify_drug_product("C4-0976", "HUGO ARTEMETHER/LUMEFANTRINE")
print("\nTest 3 (Failed Surveillance Sample):", test3)

Dataset Loaded Successfully! Total Samples: 778
Class Breakdown:
 - Genuine (0): 727
 - Suspicious/Unsatisfactory (1): 51

  CLASSIFICATION REPORT (ML BASELINE)
              precision    recall  f1-score   support

     Genuine       0.95      1.00      0.97       146
  Suspicious       1.00      0.20      0.33        10

    accuracy                           0.95       156
   macro avg       0.97      0.60      0.65       156
weighted avg       0.95      0.95      0.93       156


--- SAMPLE TEST CASES ---

Test 1 (Valid Record): {'status': 'GENUINE', 'confidence': 0.98, 'reason': 'Matches valid NAFDAC registered product in surveillance database.'}

Test 2 (Mismatched Brand): {'status': 'SUSPICIOUS', 'confidence': 0.9, 'reason': "NAFDAC Reg No exists, but is registered to ['MEPIRYL'], not 'FAKE AMARYL'."}

Test 3 (Failed Surveillance Sample): {'status': 'SUSPICIOUS', 'confidence': 1.0, 'reason': "NAFDAC Reg No found in surveillance records with flagged status: 'UNSATISFACTORY'."}
